Install Dependencies

In [7]:
pip install langchain langchain-anthropic numexpr
pip install -U langchain langchain-anthropic langchain-core

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Imports and Setup

In [9]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
from dotenv import load_dotenv
import math

load_dotenv()
print("All imports successful!")

All imports successful!


Initialize Claude

In [10]:
llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    temperature=0
)
print("Claude ready!")

Claude ready!


Define Tools

In [11]:
@tool
def calculator(expression: str) -> str:
    """Evaluates a basic math expression like '2 + 2' or '15 * 4'."""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def square_root(number: float) -> str:
    """Calculates the square root of a number."""
    try:
        result = math.sqrt(number)
        return f"The square root of {number} is {result}"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def power(base: float, exponent: float) -> str:
    """Raises base to the power of exponent."""
    try:
        result = math.pow(base, exponent)
        return f"{base} to the power of {exponent} is {result}"
    except Exception as e:
        return f"Error: {str(e)}"

tools = [calculator, square_root, power]
print(f"{len(tools)} tools ready: {[t.name for t in tools]}")

3 tools ready: ['calculator', 'square_root', 'power']


Bind tools to Claude and build agent

In [12]:
# Bind tools directly to Claude - this is the modern LangChain approach
llm_with_tools = llm.bind_tools(tools)

# Build the agent as a simple chain
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

def run_agent(question: str):
    print(f"\n🤔 Question: {question}")
    print("-" * 50)
    
    messages = [
        SystemMessage(content="You are a helpful math assistant. Use the available tools to solve math problems accurately."),
        HumanMessage(content=question)
    ]
    
    # First LLM call - decides which tool to use
    response = llm_with_tools.invoke(messages)
    messages.append(response)
    
    # Check if agent wants to use tools
    while response.tool_calls:
        for tool_call in response.tool_calls:
            print(f"🔧 Using tool: {tool_call['name']}")
            print(f"   Input: {tool_call['args']}")
            
            # Find and execute the right tool
            tool_map = {t.name: t for t in tools}
            selected_tool = tool_map[tool_call["name"]]
            tool_result = selected_tool.invoke(tool_call["args"])
            
            print(f"   Result: {tool_result}")
            
            # Add tool result back to messages
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )
        
        # Final LLM call - formulates answer using tool result
        response = llm_with_tools.invoke(messages)
        messages.append(response)
    
    print(f"\n✅ Answer: {response.content}")
    return response.content

print("Agent ready!")

Agent ready!


Test the Agent

In [13]:
# Test 1 - Basic calculation
run_agent("What is 347 multiplied by 28?")


🤔 Question: What is 347 multiplied by 28?
--------------------------------------------------
🔧 Using tool: calculator
   Input: {'expression': '347 * 28'}
   Result: 9716

✅ Answer: **347 × 28 = 9,716**


'**347 × 28 = 9,716**'

In [15]:
# Test 2 - Square root
run_agent("What is the square root of 144?")


🤔 Question: What is the square root of 144?
--------------------------------------------------
🔧 Using tool: square_root
   Input: {'number': 144}
   Result: The square root of 144.0 is 12.0

✅ Answer: The square root of **144** is **12**! 🎉

This means that 12 × 12 = 144. It's a perfect square!


"The square root of **144** is **12**! 🎉\n\nThis means that 12 × 12 = 144. It's a perfect square!"

In [16]:
# Test 3 - Multi step
run_agent("If I have a square with area 225, what is the length of each side?")


🤔 Question: If I have a square with area 225, what is the length of each side?
--------------------------------------------------
🔧 Using tool: square_root
   Input: {'number': 225}
   Result: The square root of 225.0 is 15.0

✅ Answer: The length of each side of the square is **15 units**!

This makes sense because 15 × 15 = 225, which confirms our answer. The area of a square is calculated by squaring its side length, so to reverse that, we simply take the square root of the area.


'The length of each side of the square is **15 units**!\n\nThis makes sense because 15 × 15 = 225, which confirms our answer. The area of a square is calculated by squaring its side length, so to reverse that, we simply take the square root of the area.'

In [17]:
# Test 4 - Challenge it with a multi-step problem
run_agent("What is 2 to the power of 10, and then what is the square root of that result?")


🤔 Question: What is 2 to the power of 10, and then what is the square root of that result?
--------------------------------------------------
🔧 Using tool: power
   Input: {'base': 2, 'exponent': 10}
   Result: 2.0 to the power of 10.0 is 1024.0
🔧 Using tool: square_root
   Input: {'number': 1024}
   Result: The square root of 1024.0 is 32.0

✅ Answer: Here's the full breakdown:

1. **2¹⁰ = 1024**
2. **√1024 = 32**

So, 2 to the power of 10 is **1024**, and the square root of that result is **32**! 🎉


"Here's the full breakdown:\n\n1. **2¹⁰ = 1024**\n2. **√1024 = 32**\n\nSo, 2 to the power of 10 is **1024**, and the square root of that result is **32**! 🎉"